# 02 · Training

Fine-tune the YOLO detector on the curated COCO subset.

This notebook is a thin, interactive wrapper around `scripts/train.py` / `object_tracking_app.models.detector.Detector` -- useful for Colab where you want to tweak hyperparameters and watch metrics live.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

from object_tracking_app.config.settings import get_settings
from object_tracking_app.models.detector import Detector

settings = get_settings()
settings.dataset.mode = 'demo'   # switch to 'full_subset' for a longer, higher-quality run
settings.train.epochs = 10
print(settings.train)

## Confirm the dataset subset exists

If this cell errors, run (in a terminal / a `!` cell):
```
uv run python main.py download-data --mode demo
```

In [ ]:
data_yaml = settings.resolve(settings.dataset.yolo_export.output_dir) / 'data.yaml'
assert data_yaml.exists(), f'{data_yaml} not found -- build the dataset subset first.'
print('Using dataset config:', data_yaml)

## Train

In [ ]:
detector = Detector(weights_path=settings.train.base_model, settings=settings)
result = detector.train(data_yaml=str(data_yaml))
result

## Copy the best checkpoint into artifacts/exported_models/

In [ ]:
import shutil

save_dir = Path(result['save_dir']) if result.get('save_dir') else None
if save_dir:
    best_pt = save_dir / 'weights' / 'best.pt'
    dest = settings.resolve(settings.train.checkpoint.export_best_to)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if best_pt.exists():
        shutil.copy2(best_pt, dest)
        print('Copied best checkpoint to', dest)
    else:
        print('best.pt not found under', save_dir)